In [1]:
from neuron import h
import matplotlib.pyplot as plt
import numpy as np
import plotly
import matplotlib
import pandas as pd
import plotly.graph_objects as go

In [2]:
h.load_file("main.hoc")

	1 
	1 
	1 
	1 
	0 
	1 
23 
	1 
	1 
After any change to cell geometry or nseg, be sure to invoke setpointers()
	1 
	1 
	1 
	0 
	0 
	0 
	0 
	0 
	0 
	1 
Use setstim(DEL, DUR, F0, F1, AMP, PHASE) to change latency (ms), duration (ms),
start and end frequency (Hz), amplitude (V/m) and phase (degrees) of applied
electrical field.
	0 
	0 
	0 
	0 
	0 
	0 
	0 
	0 
	1 
	1 
	1 
	0 
	0 
	0 
	0 
	0 
	0 


1.0

In [23]:
h.load_file("add_random_synapses.hoc")

	42 
	34.521743 
	3.743725e+09 
	0.14110872 
	6.474704e+08 
	41.286144 
Inserting synapse 1 
Randomly selected apical dendrite: 37 
Randomly selected location: 0.10085806 
Randomly determined distance to soma: 278.28887 
The synapse will receive input at 158 ms

Inserting synapse 2 
Randomly selected apical dendrite: 13 
Randomly selected location: 0.064537826 
Randomly determined distance to soma: 134.10369 
The synapse will receive input at 120 ms

Inserting synapse 3 
Randomly selected apical dendrite: 20 
Randomly selected location: 0.77500328 
Randomly determined distance to soma: 255.8465 
The synapse will receive input at 168 ms

Inserting synapse 4 
Randomly selected apical dendrite: 10 
Randomly selected location: 0.96113046 
Randomly determined distance to soma: 93.129193 
The synapse will receive input at 25 ms

Inserting synapse 5 
Randomly selected apical dendrite: 19 
Randomly selected location: 0.066603578 
Randomly determined distance to soma: 179.61891 
The synapse wil

1.0

In [20]:
h.changefield(360, 180, 100)

0.0

In [25]:
neuron_sections = h.SectionList(
    [sec for sec in h.allsec() if "Burst" not in str(sec) and "sField" not in str(sec) and "sElec" not in str(sec)])

ps = h.PlotShape(neuron_sections, False)
ps.show(1)

fig = ps.plot(plotly, cmap = matplotlib.colormaps["inferno"])

fig.update_layout(
    scene = dict(
        xaxis_title = "",
        yaxis_title = "",
        zaxis_title = "",
        xaxis = dict(showbackground = False, showticklabels = False),
        yaxis = dict(showbackground = False, showticklabels = False),
        zaxis = dict(showbackground = False, showticklabels = False)
    )
)

# 
# Show direction of the electric field
# 
x = h.sField.x3d(1)
y = h.sField.y3d(1)
z = h.sField.z3d(1)

end = [x, y, z]
start = [0, 0, 0]

fig.add_trace(
    go.Cone(
        x = [end[0]], y = [end[1]], z = [end[2]],
        u = [end[0] - start[0]],
        v = [end[1] - start[1]],
        w = [end[2] - start[2]],
        sizemode = "absolute",
        sizeref = 50,
        anchor = "tip",
        colorscale = [[0, "red"], [1, "red"]],
        showscale = False
    )
)

fig.add_trace(
    go.Scatter3d(
        x = [start[0], end[0]],
        y = [start[1], end[1]],
        z = [start[2], end[2]],
        mode = "lines",
        line = dict(color = "red", width = 3)
    )
)

# 
# Show the zero isopotential plane
# 

# The general equation for the plane is
# ax + by + cz = d

# Since the plane passes through (0, 0, 0), the constant d is 0.

# A nonzero vector that is orthogonal to direction vectors of
# the plane is called a normal vector to the plane.
# In this case, it is the "direction" of the electric field.

p0 = [0, 0, 0]
p1 = [h.sElec.x3d(1), h.sElec.y3d(1), h.sElec.z3d(1)]
normal = [h.sField.x3d(1), h.sField.y3d(1), h.sField.z3d(1)]

# Plane equation: n[0]*x + n[1]*y + n[2]*z = 0
a, b, c = normal

x = np.linspace(-1000, 1000, 10)
y = np.linspace(-1000, 1000, 10)
x, y = np.meshgrid(x, y)
z = (-a * x - b * y) / c

fig.add_trace(
    go.Surface(x = x, y = y, z = z, opacity = 0.25, colorscale = "gray", showscale = False)
)

# Clipping due to large plane size
fig.update_layout(
    scene = dict(
        zaxis = dict(
            range = [-350, 350]
        ),
        yaxis = dict(
            range = [-500, 500]
        ),
        xaxis = dict(
            range = [-500, 500]
        )
    )
)

#
# Show synapse locations
#
synapse_dendrites = np.array(h.synapse_dendrites)
synapse_locations = np.array(h.synapse_locations)

for dendrite_i, location in zip(synapse_dendrites, synapse_locations):
    dendrite_i = int(dendrite_i)
    i = int(h.apical_dendrite[dendrite_i].n3d() * location)
    x = h.apical_dendrite[dendrite_i].x3d(i)
    y = h.apical_dendrite[dendrite_i].y3d(i)
    z = h.apical_dendrite[dendrite_i].z3d(i)

    fig.add_trace(
        go.Scatter3d(
            x = [x],
            y = [y],
            z = [z],
            marker = dict(
                color = "red",
                size = 3
            )
        )
    )

fig.show(config = { "scrollZoom": False })